# Corr_labpks: angular kernel sensing and Chebyshev-shell NEEP

LABP-KS has four directions (up, right, down, left), periodic exclusion, and uniform spatial sensing within each **exclusive Chebyshev shell**. For a particle with direction \(c_i\),
\[
q_{i,k}=\frac{1+\chi}{2}N_{i,k}^{c_i+1}+\frac{1-\chi}{2}N_{i,k}^{c_i-1},\qquad
G_{i,k}=\beta_k[1-e^{-q_{i,k}/n_*}],\qquad
w_{i,+}=w_-+(w_+^0-w_-)e^{\sum_k G_{i,k}}.
\]
Direction indices are modulo four. \(N_{i,k}^{d}\) counts neighbors with direction \(d\) anywhere on shell \(k\). Same-direction and opposite-direction neighbors do not enter sensing. Only forward propulsion changes; backward, lateral, and both quarter-turn rates remain fixed. Blocked hops have rate zero. This defines an implementation-specific jump medium entropy, not a thermodynamic consistency claim.

The default \(\chi=1\) senses only direction \(c_i+1\). \(\chi=0\) senses both perpendicular directions equally; \(\chi=-1\) reverses the cyclic preference. `LABPKS_ANGULAR_BIAS`, `LABPKS_GAIN`, and `LABPKS_TRAIN_SEED` make separate control runs easy without changing the NN. Positive and negative bias are mirror partners: their scalar entropy rates and radial shell spectra need not change sign. Even \(\chi=0\) remains active and irreversible. At fixed density, changing \(\chi\) can also change the distribution of \(q\) and the mean saturated speed, so the comparison is not an isolated intervention on irreversibility alone.

Each training pair is exactly one **actual Gillespie event** (hop or rotation), with independent replicas. Waiting times are retained for physical-rate diagnostics and never weight the training loss. Exact event log-rate ratios are diagnostic labels only. Training uses the unchanged `MultiScaleShellForceCNEEP2D` architecture and four orientation occupancy channels.

`LABPKS_BACKEND=auto|cpu|cuda` chooses simulation independently of PyTorch training. CPU threads or bounded CUDA batches run independent exact Gillespie clocks. `LABPKS_CUDA_BATCH_SIZE` caps replicas per batch; `LABPKS_WORKERS` controls CPU workers. Explicit CUDA requests fail if unavailable. `LABPKS_SMOKE=1` executes a small end-to-end check; it establishes neither stationarity nor spectrum recovery. Run cells in order.


In [ ]:
import os
import sys
import json
import math
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import asdict
from types import SimpleNamespace

def find_project_root():
    override = os.environ.get("CNEEP_ROOT")
    starts = [Path(override).expanduser()] if override else [Path.cwd()]
    for start in starts:
        for parent in (start, *start.parents):
            for candidate in (parent, parent / "CNEEP_v2"):
                if (candidate / "models" / "NEEP_ShellForce_2D.py").is_file() and (candidate / "data" / "labp_ks").is_dir():
                    return candidate.resolve()
    raise FileNotFoundError("Open this notebook inside CNEEP_v2 or set CNEEP_ROOT to that project directory.")

ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from tqdm.auto import trange
from data.labp_ks import LABPKSConfig, LABPKSEventResult, simulate_event_ensemble, encode_observations, simulation_backend
from models.NEEP_ShellForce_2D import MultiScaleShellForceCNEEP2D

SMOKE = os.environ.get("LABPKS_SMOKE", "0") == "1"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
torch.manual_seed(2026)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(2026)
print("Project:", ROOT)
print("Training device:", DEVICE, "| smoke:", SMOKE)


## 1. Physics, independent replicas and transitions per trajectory

Keep the physics, ensemble sizes and optimizer settings in the configuration cell. `N_EVENTS` controls the number of actual transitions in each trajectory, so `N_FRAMES = N_EVENTS + 1`. Every adjacent pair is exactly one allowed hop or rotation; no empty fixed-time intervals or multi-event pairs occur.

Each replica first undergoes physical burn-in for `BURN_TIME`, followed by `EVENT_WARMUP` discarded transitions. The first discarded event moves collection onto an event boundary; the remaining event warm-up helps relax the jump chain but does not prove stationarity. Waiting times are saved, and physical durations differ between replicas. Four one-hot channels are constructed only for the current minibatch.


In [ ]:
ANGULAR_BIAS = float(os.environ.get("LABPKS_ANGULAR_BIAS", "1"))
SENSING_GAIN = float(os.environ.get("LABPKS_GAIN", "1"))
SENSING_SCALE = 1.0
SHELL_WEIGHTS = (0.0, SENSING_GAIN, 0.0)  # beta_2, beta_3, beta_4; one active shell initially

config = LABPKSConfig(
    lattice_size=10 if SMOKE else 30,
    density=0.15,
    forward_rate=8.0,
    backward_rate=0.1,
    lateral_rate=0.1,
    rotation_rate=0.01,
    shell_weights=SHELL_WEIGHTS,
    sensing_scale=SENSING_SCALE, angular_bias=ANGULAR_BIAS,
)
TRAIN_M, VAL_M, TEST_M = (2, 2, 2) if SMOKE else (100, 20, 10)
N_EVENTS = 16 if SMOKE else 1000  # pairs per trajectory; keep the dataset size fixed
N_FRAMES = N_EVENTS + 1
BURN_TIME = 0.1 if SMOKE else 5000.0
EVENT_WARMUP = 2 if SMOKE else 128
BURN_BACKEND = os.environ.get("LABPKS_BACKEND", "auto").strip().lower()
RESOLVED_BURN_BACKEND = simulation_backend(BURN_BACKEND)
EVENT_BACKEND = RESOLVED_BURN_BACKEND
CUDA_BATCH_SIZE = int(os.environ.get("LABPKS_CUDA_BATCH_SIZE", "1024"))
WORKERS = int(os.environ["LABPKS_WORKERS"]) if "LABPKS_WORKERS" in os.environ else None  # None uses available CPUs
SPLIT_SEEDS = {"train": 12031, "validation": 23041, "test": 34057}
CACHE_DATA = not SMOKE
FORCE_RESIMULATE = False

# Each shell branch only sees that exclusive Chebyshev shell.
R = len(config.shell_weights) + 1
AREA = config.lattice_size**2
MODEL_MAX_DISTANCE = min(R + 1, (config.lattice_size - 1) // 2)
opt = SimpleNamespace(
    n_components=4, n_channel=8 if SMOKE else 64, n_hidden=4,
    max_distance=MODEL_MAX_DISTANCE, include_k0=True,
    positional=False, beta=0.0, shell_center_mode="relative_only",
    shell_relative_mode="learned_absolute", shell_force_bias=True,
    shell_weight_normalization="none", shell_force_activation="elu",
)
N_ITER = 3 if SMOKE else 10000
BATCH_SIZE = 8 if SMOKE else 512
EVAL_BATCH_SIZE = 16 if SMOKE else 512
RECORD_EVERY = 1 if SMOKE else 50
LEARNING_RATE = 3e-3
WEIGHT_DECAY = 1e-3
ALPHA = 0
CLIP_NORM = 10.0
TRAIN_SEED = int(os.environ.get("LABPKS_TRAIN_SEED", "1"))

if min(TRAIN_M, VAL_M, TEST_M) < 2:
    raise ValueError("Use at least two independent trajectories in each split; TRAIN_M=1 is deliberately disallowed.")
assert len(set(SPLIT_SEEDS.values())) == 3
if N_EVENTS < 2 or EVENT_WARMUP < 1:
    raise ValueError("Use at least two recorded events and one discarded initial event.")
assert MODEL_MAX_DISTANCE < config.lattice_size / 2
RESULT_BASE = Path(os.environ.get("LABPKS_OUTPUT_DIR", str(ROOT / "results"))).expanduser()
RESULT_DIR = RESULT_BASE / ("Corr_labpks_smoke" if SMOKE else "Corr_labpks") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RESULT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = Path(os.environ.get("LABPKS_CACHE_DIR", str(ROOT / "data" / "labp_ks" / "cache"))).expanduser()
SANITY_SOURCE_PATH = ROOT / "data" / "labp_ks" / "sanity_check_labpks.ipynb"
SANITY_SOURCE_HASH = hashlib.sha256(SANITY_SOURCE_PATH.read_bytes()).hexdigest()
print(json.dumps(asdict(config), indent=2))
print(f"Physical burn-in: {RESOLVED_BURN_BACKEND}; event recording: {EVENT_BACKEND}; training: {DEVICE}")
print(f"CPU workers={WORKERS if WORKERS is not None else 'auto'}; CUDA replica batch size={CUDA_BATCH_SIZE}")
print(f"Independent replicas: train={TRAIN_M}, validation={VAL_M}, test={TEST_M}")
print(f"Train dataset: {TRAIN_M} x {N_EVENTS} pairs ({N_FRAMES} frames/trajectory); burn={BURN_TIME:g}")
print(f"Chebyshev k=0..{MODEL_MAX_DISTANCE}; discarded event warm-up={EVENT_WARMUP}")
print("Each training pair is one actual transition; no fixed delta_t or sampling calibration.")
print(f"Angular bias chi={ANGULAR_BIAS:g}; sensing gain={SENSING_GAIN:g}; forward propulsion only.")
print("Results:", RESULT_DIR)


## 2. Simulate exact event pairs or load matching event caches

Cache metadata explicitly records `sampling_mode="event"`, event count, discarded warm-up, physical burn-in backend, event recording backend, source hashes, and seeds. Fixed-time or different-backend caches cannot be loaded into this workflow. States are compact int8 arrays; reference entropy maps are stored as sparse source/target endpoints and materialized only for held-out plots. CPU and CUDA use different RNGs, so seeded paths can differ across backends; changing CUDA replica batch size preserves its paths.

Equal weighting of event pairs samples the **embedded jump chain**, whose stationary state distribution is proportional to the physical stationary probability times the escape rate. This is intentionally different from uniform physical-time sampling. For a stationary jump chain the forward/reverse joint event probability ratio is the stationary flux ratio: the escape-rate factors cancel. Its score includes a stationary-distribution boundary term in addition to `log(w_forward/w_reverse)`; an individual score is not a supervised medium-entropy label.

For rates, use `sum(score) / sum(waiting_times)` within each replica, or the pooled ratio of sums. Never average `score / waiting_time`. Finite event runs and incomplete relaxation can still affect estimates.


In [ ]:
def source_hash(directory):
    digest = hashlib.sha256()
    for path in sorted(directory.glob("*.py")):
        digest.update(path.name.encode())
        digest.update(path.read_bytes())
    return digest.hexdigest()

SIMULATOR_HASH = source_hash(ROOT / "data" / "labp_ks")
cuda_source_path = ROOT / "data" / "labp_ks" / "_gillespie_cuda.py"
CUDA_SOURCE_HASH = hashlib.sha256(cuda_source_path.read_bytes()).hexdigest() if cuda_source_path.is_file() else None
ARRAY_FIELDS = tuple(LABPKSEventResult.__dataclass_fields__)
cache_ids = {}

def simulate_split(name, count):
    request = {
        "schema": 4, "sampling_mode": "event", "config": asdict(config),
        "n_trajectories": count, "n_events": N_EVENTS,
        "event_warmup": EVENT_WARMUP, "burn_time": BURN_TIME,
        "seed": SPLIT_SEEDS[name], "simulator_sha256": SIMULATOR_HASH,
        "event_backend": EVENT_BACKEND, "burn_backend": RESOLVED_BURN_BACKEND,
        "cuda_batch_size": CUDA_BATCH_SIZE, "cuda_source_sha256": CUDA_SOURCE_HASH,
    }
    metadata = json.dumps(request, sort_keys=True, separators=(",", ":"))
    cache_id = hashlib.sha256(metadata.encode()).hexdigest()[:20]
    cache_ids[name] = cache_id
    path = CACHE_DIR / f"{name}_events_{cache_id}.npz"
    if CACHE_DATA and path.is_file() and not FORCE_RESIMULATE:
        with np.load(path, allow_pickle=False) as archive:
            if str(archive["metadata"].item()) != metadata:
                raise ValueError(f"Cache metadata mismatch: {path}")
            result = LABPKSEventResult(**{field: archive[field] for field in ARRAY_FIELDS})
        print(f"{name}: loaded {count} event trajectories from {path.name}")
    else:
        result = simulate_event_ensemble(
            config, n_trajectories=count, n_events=N_EVENTS, burn_time=BURN_TIME,
            seed=SPLIT_SEEDS[name], workers=WORKERS, progress=not SMOKE,
            backend=EVENT_BACKEND, burn_backend=RESOLVED_BURN_BACKEND, batch_size=CUDA_BATCH_SIZE,
            event_warmup=EVENT_WARMUP,
        )
        if CACHE_DATA:
            CACHE_DIR.mkdir(parents=True, exist_ok=True)
            temporary = path.with_suffix(".partial.npz")
            np.savez_compressed(temporary, metadata=np.asarray(metadata),
                                **{field: np.asarray(getattr(result, field)) for field in ARRAY_FIELDS})
            temporary.replace(path)
    assert result.states.shape == (count, N_FRAMES, config.lattice_size, config.lattice_size)
    assert result.times.shape == (count, N_FRAMES)
    assert result.medium_ep.shape == (count, N_EVENTS)
    np.testing.assert_allclose(np.diff(result.times, axis=1), result.waiting_times, rtol=1e-8, atol=1e-12)
    np.testing.assert_allclose(result.shell_ep.sum(axis=-1), result.medium_ep, atol=1e-9)
    assert np.isfinite(result.medium_ep).all()
    assert np.isfinite(result.waiting_times).all() and np.all(result.waiting_times >= 0)
    assert np.all(result.times[:, -1] > 0)
    assert np.all((result.event_types >= 0) & (result.event_types < 6))
    # Verify one change for a rotation, two changed sites for a hop; use one
    # replica at a time to avoid allocating a full-ensemble boolean video.
    for replica in range(count):
        changes = np.count_nonzero(result.states[replica, 1:] != result.states[replica, :-1], axis=(-2, -1))
        np.testing.assert_array_equal(changes, np.where(result.event_types[replica] < 4, 2, 1))
    return result

results = {name: simulate_split(name, count) for name, count in
           (("train", TRAIN_M), ("validation", VAL_M), ("test", TEST_M))}
all_seeds = np.concatenate([s for result in results.values() for s in (result.seeds, result.burn_seeds)])
assert len(np.unique(all_seeds)) == len(all_seeds), "Replica seeds overlap."
videos = {name: torch.from_numpy(result.states) for name, result in results.items()}
train_video, val_video, test_video = (videos[name] for name in ("train", "validation", "test"))
sampling_diagnostics = {}
for name, video in videos.items():
    assert video.dtype == torch.int8 and video.ndim == 4
    result = results[name]
    durations = result.waiting_times.sum(axis=1)
    sampling_diagnostics[name] = {
        "events_per_replica": N_EVENTS,
        "hop_fraction": float(result.hop_counts.mean()),
        "rotation_fraction": float(result.rotation_counts.mean()),
        "mean_waiting_time": float(result.waiting_times.mean()),
        "waiting_time_quantiles_0_50_90_99_100": np.quantile(result.waiting_times, [0, .5, .9, .99, 1]).tolist(),
        "physical_durations": durations.tolist(),
        "pooled_event_rate": float(result.event_types.size / durations.sum()),
    }
    print(name, tuple(video.shape), "| event sampling:", sampling_diagnostics[name])


## 3. Snapshots and relaxation checks

The same trajectory is shown at three event indices, with its own elapsed physical times. Event-weighted adjacent-site occupancy correlation and medium entropy rates in the first/second event halves are simple drift checks, not a proof of stationarity. Compare longer burn times and larger systems before interpreting an apparent cluster as MIPS.

In [ ]:
direction_cmap = ListedColormap(["white", "#377eb8", "#e41a1c", "#4daf4a", "#984ea3"])
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5), constrained_layout=True)
for ax, frame in zip(axes, (0, N_FRAMES // 2, N_FRAMES - 1)):
    image = ax.imshow(results["train"].states[0, frame] + 1, cmap=direction_cmap,
                      vmin=-0.5, vmax=4.5, interpolation="nearest")
    ax.set_title(f"Train replica 0, event={frame}, t={results['train'].times[0, frame]:.3g}")
    ax.set_xticks([]); ax.set_yticks([])
bar = fig.colorbar(image, ax=axes, ticks=range(5), shrink=0.85)
bar.ax.set_yticklabels(["empty", "up", "right", "down", "left"])
fig.savefig(RESULT_DIR / "direction_snapshots.png", dpi=150)
plt.show()

def mean_sem(values):
    values = np.asarray(values, dtype=np.float64)
    return float(values.mean()), float(values.std(ddof=1) / np.sqrt(len(values)))

def nearest_correlation(states):
    # Work on one replica at a time; avoid full-ensemble float64 lattice temporaries.
    covariance = np.empty(states.shape[:2], dtype=np.float64)
    for replica, state in enumerate(states):
        occupied = state >= 0
        horizontal = (occupied & np.roll(occupied, 1, axis=-1)).mean(axis=(-2, -1))
        vertical = (occupied & np.roll(occupied, 1, axis=-2)).mean(axis=(-2, -1))
        rho = occupied.mean(axis=(-2, -1))
        covariance[replica] = 0.5 * (horizontal + vertical) - rho**2
    return covariance

drift_diagnostics = {}
for name, result in results.items():
    mid = (N_FRAMES - 1) // 2
    first = result.medium_ep[:, :mid].sum(axis=1) / result.waiting_times[:, :mid].sum(axis=1)
    second = result.medium_ep[:, mid:].sum(axis=1) / result.waiting_times[:, mid:].sum(axis=1)
    corr = nearest_correlation(result.states)
    drift_diagnostics[name] = {
        "medium_rate_first_half": mean_sem(first),
        "medium_rate_second_half": mean_sem(second),
        "nearest_covariance_first_half": float(corr[:, :N_FRAMES // 2].mean()),
        "nearest_covariance_second_half": float(corr[:, N_FRAMES // 2:].mean()),
    }
print(json.dumps(drift_diagnostics, indent=2))

## 4. Ensemble-aware pair sampling and unsupervised training

A training sample selects a replica and one adjacent event pair, uniformly over actual transitions. Validation evaluates every pair of its own replicas; no transition is formed across replica boundaries.

The model API returns a **spatial mean**, so this notebook multiplies branch outputs by the lattice area to obtain extensive branch entropy increments. Their sum is the dimensionless total score used in alpha-NEEP. The model's raw maps are local entropy increments: summing them over sites and branches gives exactly that total score. This convention keeps local learned forces independent of the lattice area. No holding-time factor enters the objective. Physical rates are computed afterward as the sum of increments divided by the sum of actual waiting times.

The final contraction uses increments of the four orientation occupancy channels. The outer branch sees only its exclusive Chebyshev shell. Final force heads start at zero, giving the exact zero-score baseline while hidden features retain their random initialization. This architecture is antisymmetric by construction, but exact recovery of every path contribution is not assumed.

In [ ]:
torch.manual_seed(TRAIN_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(TRAIN_SEED)
pair_rng = torch.Generator(device="cpu").manual_seed(TRAIN_SEED + 1)
model = MultiScaleShellForceCNEEP2D(opt).to(DEVICE)
# Begin at the zero-score baseline, avoiding large random extensive scores.
for branch in model.branches:
    torch.nn.init.zeros_(branch.force_head[-1].weight)
    if branch.force_head[-1].bias is not None:
        torch.nn.init.zeros_(branch.force_head[-1].bias)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

def pair_batch(video, replica_index, time_index):
    packed = torch.stack((video[replica_index, time_index],
                          video[replica_index, time_index + 1]), dim=1).to(DEVICE)
    directions = torch.arange(4, device=DEVICE, dtype=packed.dtype).view(1, 1, 4, 1, 1)
    return (packed.unsqueeze(2) == directions).to(torch.float32)

def random_pairs(video, batch_size):
    replica = torch.randint(video.shape[0], (batch_size,), generator=pair_rng)
    time = torch.randint(video.shape[1] - 1, (batch_size,), generator=pair_rng)
    return pair_batch(video, replica, time)

def ordered_pairs(video, batch_size):
    per_replica = video.shape[1] - 1
    for start in range(0, video.shape[0] * per_replica, batch_size):
        index = torch.arange(start, min(start + batch_size, video.shape[0] * per_replica))
        yield index // per_replica, index % per_replica

def alpha_neep_terms(score):
    if ALPHA == 0:
        return -score + torch.expm1(-score)
    if not -1 < ALPHA < 0:
        raise ValueError("Choose ALPHA=0 or -1 < ALPHA < 0.")
    return -torch.expm1(ALPHA * score) / ALPHA + torch.expm1(-(1 + ALPHA) * score) / (1 + ALPHA)

def branch_scores(batch):
    return model(batch) * AREA  # API spatial mean -> extensive branch increment

@torch.no_grad()
def evaluate_objective(video):
    model.eval()
    total, count = 0.0, 0
    for replica, time in ordered_pairs(video, EVAL_BATCH_SIZE):
        score = branch_scores(pair_batch(video, replica, time)).sum(dim=1)
        total += alpha_neep_terms(score).double().sum().item()
        count += len(score)
    return total / count

with torch.no_grad():
    check = pair_batch(train_video, torch.tensor([0, 1]), torch.tensor([0, 0]))
    expected = encode_observations(results["train"].states[[0, 1], :2]).to(DEVICE)
    torch.testing.assert_close(check, expected, rtol=0, atol=0)
    branch = branch_scores(check)
    assert torch.count_nonzero(branch).item() == 0, "Zero force heads must give the exact zero-score baseline."
    raw_maps = model(check, return_maps=True)
    assert branch.shape == (2, MODEL_MAX_DISTANCE + 1)
    torch.testing.assert_close(branch, raw_maps.sum(dim=(-2, -1)), atol=2e-5, rtol=2e-5)
    torch.testing.assert_close(branch_scores(check.flip(1)), -branch, atol=2e-5, rtol=2e-5)
    torch.testing.assert_close(
        raw_maps.sum(dim=(1, 2, 3)),
        branch.sum(dim=1), atol=2e-5, rtol=2e-5,
    )
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print("Pair boundaries, map normalization, and reversal checks passed.")

In [ ]:
history = {"iteration": [0], "train": [], "validation": []}
# All final force heads are zero, so every score and its objective are exactly zero.
# Avoid a redundant full training-set scan just to recover this baseline.
initial_train = initial_val = 0.0
if not (math.isfinite(initial_train) and math.isfinite(initial_val)):
    raise FloatingPointError("Nonfinite initial training or validation objective.")
history["train"].append(initial_train)
history["validation"].append(initial_val)
best_val, best_iteration = initial_val, 0
checkpoint_path = RESULT_DIR / "best_model.pt"

def save_checkpoint(iteration, val_loss):
    torch.save({
        "state_dict": model.state_dict(), "model_options": vars(opt),
        "iteration": iteration, "validation_loss": val_loss,
        "physics": asdict(config), "sampling_mode": "event", "n_events": N_EVENTS,
        "event_warmup": EVENT_WARMUP, "burn_time": BURN_TIME,
        "sanity_source_sha256": SANITY_SOURCE_HASH,
        "cache_ids": dict(cache_ids), "simulator_sha256": SIMULATOR_HASH,
        "event_backend": EVENT_BACKEND, "burn_backend": RESOLVED_BURN_BACKEND,
        "cuda_source_sha256": CUDA_SOURCE_HASH,
        "train_seed": TRAIN_SEED, "alpha": ALPHA,
        "score_convention": "spatial_mean_times_area", "lattice_area": AREA,
        "initialization": "zero final force heads; random hidden features",
    }, checkpoint_path)

save_checkpoint(0, initial_val)
recent_losses = []
progress = trange(1, N_ITER + 1, desc="ensemble alpha-NEEP", disable=SMOKE)
for iteration in progress:
    model.train()
    optimizer.zero_grad(set_to_none=True)
    score = branch_scores(random_pairs(train_video, BATCH_SIZE)).sum(dim=1)
    loss = alpha_neep_terms(score).mean()
    if not torch.isfinite(loss):
        raise FloatingPointError("Nonfinite alpha-NEEP loss; inspect score tails and reduce the learning rate.")
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM, error_if_nonfinite=True)
    optimizer.step()
    recent_losses.append(float(loss.detach()))
    if iteration % RECORD_EVERY == 0 or iteration == N_ITER:
        val_loss = evaluate_objective(val_video)
        if not math.isfinite(val_loss):
            raise FloatingPointError("Nonfinite validation loss.")
        history["iteration"].append(iteration)
        history["train"].append(float(np.mean(recent_losses)))
        history["validation"].append(val_loss)
        recent_losses.clear()
        if val_loss < best_val:
            best_val, best_iteration = val_loss, iteration
            save_checkpoint(iteration, val_loss)
        progress.set_postfix(validation=f"{val_loss:.4g}", best=best_iteration)

# This checkpoint was created locally by this run and only contains tensors/basic metadata.
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
model.load_state_dict(checkpoint["state_dict"])
model.eval()
print(f"Best iteration={best_iteration}, validation objective={best_val:.6g}, zero-score baseline=0")
if best_val >= -1e-6:
    warnings.warn("Validation did not improve over the zero-score baseline; interpret plots as diagnostics of unresolved learning.")
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
ax.plot(history["iteration"], history["train"], ".-", label="training")
ax.plot(history["iteration"], history["validation"], ".-", label="independent validation")
ax.axhline(0, color="k", lw=0.8)
ax.set(xlabel="iteration", ylabel="alpha-NEEP objective")
ax.legend(); ax.grid(alpha=0.2)
fig.savefig(RESULT_DIR / "training_objective.png", dpi=150)
plt.show()

## 5. Held-out event irreversibility and microscopic medium entropy

Each pair now contains one complete actual transition, so there is no hidden multi-event path between its endpoints. The unsupervised optimum is the forward/reverse **joint event probability ratio**, including the stationary-distribution boundary term. Its event-average agrees with the average microscopic medium entropy when the jump chain is stationary and the learned score is exact. This does not guarantee that the current shell-force model represents that score, or that a finite dataset is sufficient.

Report entropy per event and entropy per physical time separately. Rate conversion divides each trajectory's summed entropy by its own total waiting time. Error bars use independent trajectory summaries. The cumulative plot uses event index because physical event times differ between replicas; it never averages unequal clocks as though they were a shared time grid.


In [ ]:
@torch.no_grad()
def predict_ensemble(video):
    model.eval()
    shape = (video.shape[0], video.shape[1] - 1)
    total_pairs = shape[0] * shape[1]
    branch_output = np.empty((total_pairs, MODEL_MAX_DISTANCE + 1), dtype=np.float32)
    local_output = np.empty((total_pairs, config.lattice_size, config.lattice_size), dtype=np.float32)
    offset = 0
    for replica, time in ordered_pairs(video, EVAL_BATCH_SIZE):
        batch = pair_batch(video, replica, time)
        raw = model(batch, return_maps=True)
        if not torch.isfinite(raw).all():
            raise FloatingPointError("Nonfinite held-out prediction.")
        branches = raw.sum(dim=(-2, -1))
        local = raw.sum(dim=1)
        torch.testing.assert_close(local.sum(dim=(-2, -1)), branches.sum(dim=1), atol=2e-5, rtol=2e-5)
        end = offset + len(batch)
        branch_output[offset:end] = branches.cpu().numpy()
        local_output[offset:end] = local.cpu().numpy()
        offset = end
    assert offset == total_pairs
    return (branch_output.reshape(*shape, MODEL_MAX_DISTANCE + 1),
            local_output.reshape(*shape, config.lattice_size, config.lattice_size))

pred_branches, pred_local = predict_ensemble(test_video)
pred_total = pred_branches.sum(axis=-1)
reference = results["test"]
true_total = reference.medium_ep
true_local = reference.medium_ep_maps
np.testing.assert_allclose(pred_local.sum(axis=(-2, -1)), pred_total, rtol=2e-5, atol=2e-5)
duration = reference.waiting_times.sum(axis=1)
true_replica_rates = true_total.sum(axis=1) / duration
pred_replica_rates = pred_total.sum(axis=1) / duration
rate_summary = {
    "exact_model_medium_ep_rate_mean_sem": mean_sem(true_replica_rates),
    "learned_event_score_rate_mean_sem": mean_sem(pred_replica_rates),
    "exact_medium_entropy_per_event_mean_sem": mean_sem(true_total.mean(axis=1)),
    "learned_entropy_per_event_mean_sem": mean_sem(pred_total.mean(axis=1)),
    "pooled_exact_medium_ep_rate": float(true_total.sum() / duration.sum()),
    "pooled_learned_event_score_rate": float(pred_total.sum() / duration.sum()),
    "rate_difference_mean_sem": mean_sem(pred_replica_rates - true_replica_rates),
}
test_objective = evaluate_objective(test_video)
print(json.dumps(rate_summary, indent=2))
print(f"Held-out alpha-NEEP objective={test_objective:.6g}; zero-score baseline=0.")
if pred_total.mean() <= 0:
    warnings.warn("Held-out mean score is nonpositive; positive observed irreversibility was not resolved.")

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
event_index = np.arange(1, N_EVENTS + 1)
true_cumulative = true_total.cumsum(axis=1)
pred_cumulative = pred_total.cumsum(axis=1)
for values, label in ((true_cumulative, "exact model medium"), (pred_cumulative, "learned event score")):
    average = values.mean(axis=0)
    error = values.std(axis=0, ddof=1) / np.sqrt(TEST_M)
    axes[0].plot(event_index, average, label=label)
    axes[0].fill_between(event_index, average - error, average + error, alpha=0.2)
axes[0].set(xlabel="actual transition index", ylabel="cumulative full-system entropy")
axes[0].legend(); axes[0].grid(alpha=0.2)
replicas = np.arange(TEST_M)
axes[1].plot(replicas, true_replica_rates, "o", label="exact model medium")
axes[1].plot(replicas, pred_replica_rates, "x", label="learned event score")
axes[1].set(xlabel="independent test replica", ylabel="entropy / physical time")
axes[1].set_xticks(replicas)
axes[1].legend(); axes[1].grid(alpha=0.2)
fig.savefig(RESULT_DIR / "held_out_entropy_rates.png", dpi=150)
plt.show()

## 6. Signed shell spectrum and local maps

The simulator and the NN both inspect exclusive Chebyshev shells, but their decompositions have different meanings. The exact rate diagnostic assigns \(\log(w_+^0/w_-)\) to \(k=0\). Since \(w_+=w_-+(w_+^0-w_-)e^{\sum_kG_k}\), the effective log boost is not simply \(\sum_kG_k\): it is allocated among interaction shells **in proportion to their nonnegative \(G_k\)**. This convention sums exactly to the event log-rate ratio but is not a unique physical localization or a shell-wise training target.

Learned branch contributions are fitted jointly, may be signed, and may redistribute across shells. A hop probes two endpoints; the extra outer branch is therefore not evidence of an additional simulator interaction. Exact local maps split hop entropy symmetrically between endpoints, whereas learned maps use convolution centers.

Angular asymmetry introduces a nonsymmetric coupling between orientation channels. In the linear shell-force limit, the symmetric part of a shell operator contributes a state-function difference, which telescopes on a closed trajectory; an antisymmetric part can instead support a cycle contribution. This is a representability motivation, not a guarantee that the actual forward-only rate law or its stationary boundary term can be represented, or that training will recover a prescribed shell spectrum. Nonlinear branches, exclusion, finite data, and the changed steady state all affect the outcome. In particular, \(\chi=0\) need not have zero outer-shell score.

Compare independent training seeds and \(\chi=0,0.5,1\), a no-sensing control (`LABPKS_GAIN=0`), and the mirror control \(\chi=-1\). Test a shifted active shell as a separate range control. The diagnostic below removes learned outputs without retraining; it measures dependence of this checkpoint on its branches, not the necessity of that shell for the entire model family.


In [ ]:
distances = np.arange(MODEL_MAX_DISTANCE + 1)
learned_replica_shell_rates = pred_branches.sum(axis=1) / duration[:, None]
exact_replica_shell_rates = np.zeros_like(learned_replica_shell_rates, dtype=np.float64)
exact_width = reference.shell_ep.shape[-1]
exact_replica_shell_rates[:, :exact_width] = reference.shell_ep.sum(axis=1) / duration[:, None]
learned_shell_mean = learned_replica_shell_rates.mean(axis=0)
learned_shell_sem = learned_replica_shell_rates.std(axis=0, ddof=1) / np.sqrt(TEST_M)
exact_shell_mean = exact_replica_shell_rates.mean(axis=0)
exact_shell_sem = exact_replica_shell_rates.std(axis=0, ddof=1) / np.sqrt(TEST_M)
np.testing.assert_allclose(exact_shell_mean.sum(), true_replica_rates.mean(), atol=1e-9)
np.testing.assert_allclose(learned_shell_mean.sum(), pred_replica_rates.mean(), rtol=2e-5, atol=2e-5)

fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
ax.bar(distances - 0.19, exact_shell_mean, width=0.38, yerr=exact_shell_sem,
       capsize=3, label="exact log-rate diagnostic allocation")
ax.bar(distances + 0.19, learned_shell_mean, width=0.38, yerr=learned_shell_sem,
       capsize=3, label="learned branch decomposition")
ax.axhline(0, color="k", lw=0.8)
ax.set(xlabel="exclusive Chebyshev shell k", ylabel="signed entropy / physical time",
       title="Different decompositions; bars are not shell-wise target labels")
ax.set_xticks(distances); ax.legend(); ax.grid(axis="y", alpha=0.2)
fig.savefig(RESULT_DIR / "chebyshev_shell_spectrum.png", dpi=150)
plt.show()

replica_index = 0
frame_index = (N_FRAMES - 1) // 2
local_reference = true_local[replica_index, frame_index]
local_prediction = pred_local[replica_index, frame_index]
limit = max(float(np.abs(local_reference).max()), float(np.abs(local_prediction).max()), 1e-8)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5), constrained_layout=True)
axes[0].imshow(reference.states[replica_index, frame_index] >= 0, cmap="Greys", vmin=0, vmax=1)
axes[0].set_title("Held-out occupancy")
for ax, field, label in zip(axes[1:], (local_reference, local_prediction),
                           ("Exact symmetric-endpoint map", "Learned convolution-center map")):
    im = ax.imshow(field, cmap="RdBu_r", vmin=-limit, vmax=limit)
    ax.set_title(label)
    fig.colorbar(im, ax=ax, shrink=0.8)
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
fig.savefig(RESULT_DIR / "local_entropy_maps.png", dpi=150)
plt.show()


## 7. Held-out sensitivity to outer branches

Remove selected outputs from the frozen checkpoint and reevaluate the same unsupervised objective. A positive `objective_increase` means the removal worsens this checkpoint's objective. It is not a shell entropy label or a causal/identifiability guarantee. Report validation and untouched test replicas separately; use independent training seeds for reproducibility. The mean objective is per event and is not divided by waiting times.


In [ ]:
@torch.no_grad()
def evaluate_branch_drops(video):
    shells = np.arange(MODEL_MAX_DISTANCE + 1)
    keep = {"full": np.ones(len(shells), dtype=bool), "drop_all_outer_k_ge_2": shells < 2}
    for k in range(2, MODEL_MAX_DISTANCE + 1):
        keep[f"drop_k{k}"] = shells != k
    mask_tensor = torch.as_tensor(np.stack(list(keep.values())), dtype=torch.float32, device=DEVICE)
    replica_sums = np.zeros((video.shape[0], len(keep)), dtype=np.float64)
    replica_counts = np.zeros(video.shape[0], dtype=np.int64)
    for replica, time in ordered_pairs(video, EVAL_BATCH_SIZE):
        scores = branch_scores(pair_batch(video, replica, time)) @ mask_tensor.T
        terms = alpha_neep_terms(scores).cpu().numpy()
        if not np.isfinite(terms).all():
            raise FloatingPointError("Nonfinite branch-drop objective; inspect score tails.")
        np.add.at(replica_sums, replica.numpy(), terms)
        np.add.at(replica_counts, replica.numpy(), 1)
    means = replica_sums / replica_counts[:, None]
    baseline = means[:, 0]
    return {name: {"objective_mean_sem": mean_sem(means[:, index]),
                   "objective_increase_mean_sem": mean_sem(means[:, index] - baseline),
                   "replica_objectives": means[:, index].tolist()}
            for index, name in enumerate(keep)}

branch_ablations = {"validation": evaluate_branch_drops(val_video),
                    "test": evaluate_branch_drops(test_video)}
print(json.dumps(branch_ablations, indent=2))


## 8. Save reproducible diagnostics

The summary retains all physics including angular bias, exact-event sampling, simulator/source hashes, seeds, cache keys, checkpoint, signed branch spectrum, and branch-drop objectives. Run controls separately with the same dataset sizes, physical rates, gain and optimizer settings. The final cell reads already-completed runs; it never launches additional production training. A branch-drop objective is evaluated on the frozen model and cannot replace retraining a restricted model.


In [ ]:
summary = {
    "physics": asdict(config),
    "simulation": {
        "algorithm": "exact Gillespie", "sampling_mode": "event",
        "n_frames": N_FRAMES, "n_events": N_EVENTS, "event_warmup": EVENT_WARMUP,
        "burn_time": BURN_TIME,
        "physical_durations": {name: result.waiting_times.sum(axis=1).tolist() for name, result in results.items()},
        "workers": WORKERS, "split_seeds": SPLIT_SEEDS,
        "burn_backend_request": BURN_BACKEND, "burn_backend": RESOLVED_BURN_BACKEND,
        "event_backend": EVENT_BACKEND,
        "cuda_batch_size": CUDA_BATCH_SIZE,
        "replicas": {"train": TRAIN_M, "validation": VAL_M, "test": TEST_M},
        "replica_seeds": {name: np.asarray(result.seeds).tolist() for name, result in results.items()},
    },
    "training": {
        "model": "MultiScaleShellForceCNEEP2D", "options": vars(opt),
        "iterations": N_ITER, "batch_size": BATCH_SIZE, "seed": TRAIN_SEED,
        "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
        "optimizer": "AdamW", "clip_norm": CLIP_NORM,
        "record_every": RECORD_EVERY, "eval_batch_size": EVAL_BATCH_SIZE,
        "alpha": ALPHA, "best_iteration": best_iteration,
        "best_validation_objective": best_val, "test_objective": test_objective,
        "score_convention": "branch spatial means times lattice area; raw maps are local entropy increments",
        "lattice_area": AREA,
        "initialization": "zero final force heads; random hidden features",
    },
    "cache_ids": cache_ids, "simulator_sha256": SIMULATOR_HASH,
    "cuda_source_sha256": CUDA_SOURCE_HASH,
    "model_sha256": hashlib.sha256((ROOT / "models" / "NEEP_ShellForce_2D.py").read_bytes()).hexdigest(),
    "sampling_diagnostics": sampling_diagnostics,
    "sanity_source_sha256": SANITY_SOURCE_HASH,
    "rates": rate_summary, "drift_diagnostics": drift_diagnostics,
    "branch_ablations": branch_ablations,
    "angular_comparison_note": "chi changes reciprocity and sensing-count variance; mirror chi signs need not reverse scalar EP or shell scores.",
    "shell_diagnostic_convention": "baseline at k0; effective forward log boost allocated proportional to G_k; not supervised targets",
    "learned_shell_rate_mean": learned_shell_mean.tolist(),
    "learned_shell_rate_sem": learned_shell_sem.tolist(),
    "exact_shell_rate_mean": exact_shell_mean.tolist(),
    "exact_shell_rate_sem": exact_shell_sem.tolist(),
    "smoke": SMOKE,
    "interpretation": "Unweighted exact-event-pair irreversibility vs microscopic medium EP; jump-chain stationary boundary and shell representability still matter.",
}
(RESULT_DIR / "summary.json").write_text(json.dumps(summary, indent=2, allow_nan=False), encoding="utf-8")
np.savez_compressed(
    RESULT_DIR / "held_out_diagnostics.npz",
    times=reference.times, waiting_times=reference.waiting_times, event_types=reference.event_types,
    collection_start_times=reference.start_times, replica_seeds=reference.seeds,
    true_total_increment=true_total, learned_total_increment=pred_total,
    true_local_increment=true_local, learned_local_increment=pred_local,
    true_shell_increment=reference.shell_ep, learned_shell_increment=pred_branches,
    true_replica_rates=true_replica_rates, learned_replica_rates=pred_replica_rates,
    history_iteration=np.asarray(history["iteration"]),
    history_train=np.asarray(history["train"]),
    history_validation=np.asarray(history["validation"]),
)
plt.close("all")
print("Saved checkpoint and diagnostics:", RESULT_DIR)
print("Smoke complete." if SMOKE else "Training run complete; inspect sampling counts, held-out validation, and relaxation diagnostics.")


## 9. Compare completed angular-bias control runs

This table reads saved summaries only. Compare rows with the same `comparison_group`: these share physics except angular bias, simulator/model sources, dataset sizes and training settings. Seeds distinguish repeat fits; the group does not imply identical trajectories across different dynamics. Lower validation objective and positive outer-branch removal cost indicate useful learned information, while a radial branch amplitude alone does not identify sensing coefficients. Opposite angular biases should be compared statistically, without expecting an entropy sign reversal.


In [ ]:
completed = []
comparison_base = RESULT_BASE / ("Corr_labpks_smoke" if SMOKE else "Corr_labpks")
for summary_path in sorted(comparison_base.glob("*/summary.json")):
    previous = json.loads(summary_path.read_text(encoding="utf-8"))
    if "branch_ablations" not in previous:
        continue
    physics_key = dict(previous["physics"])
    chi = physics_key.pop("angular_bias")
    training_key = {key: previous["training"].get(key) for key in
                    ("options", "iterations", "batch_size", "learning_rate", "weight_decay", "alpha",
                     "optimizer", "clip_norm", "record_every", "eval_batch_size")}
    simulation_key = {key: previous["simulation"][key] for key in
                      ("n_events", "event_warmup", "burn_time", "replicas", "event_backend", "burn_backend", "split_seeds")}
    comparable = {"physics": physics_key, "training": training_key, "simulation": simulation_key,
                  "simulator": previous["simulator_sha256"], "model": previous["model_sha256"]}
    group = hashlib.sha256(json.dumps(comparable, sort_keys=True).encode()).hexdigest()[:10]
    completed.append({"run": summary_path.parent.name, "comparison_group": group,
                      "angular_bias": chi, "train_seed": previous["training"]["seed"],
                      "validation_objective": previous["training"]["best_validation_objective"],
                      "test_objective": previous["training"]["test_objective"],
                      "test_outer_drop_cost_mean_sem": previous["branch_ablations"]["test"]["drop_all_outer_k_ge_2"]["objective_increase_mean_sem"],
                      "learned_shell_rate_mean": previous["learned_shell_rate_mean"]})
print(json.dumps(completed, indent=2))
(RESULT_DIR / "completed_angular_controls.json").write_text(
    json.dumps(completed, indent=2, allow_nan=False), encoding="utf-8")
